# ⛰️ Heaps & Priority Queues — Runnable Notebook

Companion to [`README.md`](README.md) and
[`12_heaps_priority_queues_lesson.html`](12_heaps_priority_queues_lesson.html).

A **min-heap** built from scratch (sift up/down), then Python's `heapq` and two classic uses.

## 1. A min-heap from scratch

You want a bag of numbers where **the smallest is always one glance away**, and adding or removing that smallest is cheap — not a full sort every time. A **min-heap** is a complete tree packed into a list: the parent is never larger than its children, so index `0` is the minimum. `push` drops a value at the end and bubbles it **up**; `pop` hands back the root, drops the last value on top, and bubbles it **down**.

Stored as a plain list; `left=2i+1`, `right=2i+2`, `parent=(i-1)//2`.

In [1]:
def push(heap, x):
    """Add x at the end, then SIFT UP while it is smaller than its parent."""
    heap.append(x)
    i = len(heap) - 1
    while i > 0:
        parent = (i - 1) // 2
        if heap[i] < heap[parent]:              # smaller than parent -> bubble up
            heap[i], heap[parent] = heap[parent], heap[i]
            i = parent
        else:
            break                               # heap property restored

def pop(heap):
    """Remove the root (minimum). Move the last item up, then SIFT DOWN."""
    top = heap[0]
    last = heap.pop()
    if heap:
        heap[0] = last
        i, n = 0, len(heap)
        while True:
            small, l, r = i, 2*i + 1, 2*i + 2
            if l < n and heap[l] < heap[small]: small = l
            if r < n and heap[r] < heap[small]: small = r
            if small == i:                      # no smaller child -> done
                break
            heap[i], heap[small] = heap[small], heap[i]
            i = small
    return top

h = []
for x in [5, 1, 8, 3, 9, 2, 7]:
    push(h, x)
print("heap array (root is the min):", h)
print("root / peek:", h[0])
assert h[0] == 1                                 # the minimum is always at index 0

# popping repeatedly yields sorted order (this is heap-sort)
out = [pop(h) for _ in range(7)]
print("popped in order:", out)
assert out == sorted(out) == [1, 2, 3, 5, 7, 8, 9]

heap array (root is the min): [1, 3, 2, 5, 9, 8, 7]
root / peek: 1
popped in order: [1, 2, 3, 5, 7, 8, 9]


### Step-by-step: what this cell actually does

This is a **min-heap**: every parent is `≤` both children, so the **smallest value is always at index 0**. The tree is stored as a plain list (no node pointers). After all seven `push`es the notebook prints `[1, 3, 2, 5, 9, 8, 7]`; popping then yields `[1, 2, 3, 5, 7, 8, 9]`.

#### Array packing (why those formulas)

Because a heap is a **complete** binary tree (filled left-to-right), node `i` has:

| relation | index |
|----------|--------|
| parent | `(i - 1) // 2` |
| left child | `2*i + 1` |
| right child | `2*i + 2` |

Example — array `[1, 3, 2, 5, 9, 8, 7]`:

```
index:  0  1  2  3  4  5  6
value: [1, 3, 2, 5, 9, 8, 7]

          1          ← i=0
        /   \
      3       2      ← i=1, i=2
     / \     / \
    5   9   8   7    ← i=3,4,5,6
```

- Parent of `9` (i=4): `(4-1)//2 = 1` → `3`. Parent of `7` (i=6): `(6-1)//2 = 2` → `2`.
- Children of `3` (i=1): `2*1+1=3` and `2*1+2=4` → `5` and `9`.

`push` appends at the next empty slot (keeps the tree complete), then **sifts up**. `pop` saves the root, dumps the last item into index 0 (keeps it complete), then **sifts down**.

`i` is the current hole walking up (push) or down (pop) — not a search index over the whole array.

---

#### `push` — append, then sift up

`push(heap, x)`:

1. `heap.append(x)` — new leaf at the end.
2. `i = len(heap) - 1` — start at that leaf.
3. While `i > 0` and `heap[i] < heap[parent]`, swap with parent and set `i = parent`.
4. Stop when the new value is not smaller than its parent (or it became the root).

Demo input: `[5, 1, 8, 3, 9, 2, 7]`. Bold = the value just inserted (after it finishes bubbling).

**1. push 5** — empty list; nothing to sift.

```
array: [5]

    5
```

**2. push 1** — append `[5, 1]`. `i=1`, parent=`0`. `1 < 5` → swap → `[1, 5]`. `i=0` → stop.

```
array: [1, 5]

    1
   /
  5
```

**3. push 8** — append `[1, 5, 8]`. `i=2`, parent=`0`. `8 < 1`? No → break.

```
array: [1, 5, 8]

    1
   / \
  5   8
```

**4. push 3** — append `[1, 5, 8, 3]`. `i=3`, parent=`1`. `3 < 5` → swap → `[1, 3, 8, 5]`. Now `i=1`, parent=`0`. `3 < 1`? No.

```
array: [1, 3, 8, 5]

      1
     / \
    3   8
   /
  5
```

**5. push 9** — append `[1, 3, 8, 5, 9]`. `i=4`, parent=`1`. `9 < 3`? No.

```
array: [1, 3, 8, 5, 9]

      1
     / \
    3   8
   / \
  5   9
```

**6. push 2** — append `[1, 3, 8, 5, 9, 2]`. `i=5`, parent=`2`. `2 < 8` → swap → `[1, 3, 2, 5, 9, 8]`. `i=2`, parent=`0`. `2 < 1`? No.

```
array: [1, 3, 2, 5, 9, 8]

      1
     / \
    3   2
   / \ /
  5  9 8
```

**7. push 7** — append `[1, 3, 2, 5, 9, 8, 7]`. `i=6`, parent=`2`. `7 < 2`? No.

```
array: [1, 3, 2, 5, 9, 8, 7]

        1
       / \
      3   2
     / \ / \
    5  9 8  7
```

Root / peek is `h[0] == 1`. The list is **not** fully sorted — only the heap property holds (each parent ≤ its children). That is enough for `O(1)` peek.

---

#### `pop` — take the root, last item becomes root, then sift down

`pop(heap)`:

1. `top = heap[0]` — that is the minimum (return this later).
2. `last = heap.pop()` — remove the last leaf so the tree stays complete.
3. If the heap is now empty, return `top` (nothing to sift).
4. Else `heap[0] = last` — the old last leaf sits at the root (often too big).
5. Sift down: at `i`, look at left `2i+1` and right `2i+2`. Let `small` be `i` or whichever child is smaller. If `small == i`, stop. Else swap with that child and continue.

Each pop below shows the array **after** the last item has been moved to the root, then after sift-down finishes.

**pop → 1.** Save `1`. Last leaf `7` goes to root: `[7, 3, 2, 5, 9, 8]`. Children of `7` are `3` and `2`; smaller is `2`. Swap → `[2, 3, 7, 5, 9, 8]`. At `7`, left child is `8` (`i=5`); `8 < 7`? No → stop.

```
after: [2, 3, 7, 5, 9, 8]

      2
     / \
    3   7
   / \ /
  5  9 8
```

**pop → 2.** Last `8` to root: `[8, 3, 7, 5, 9]`. Smaller child of `8` is `3`. Swap → `[3, 8, 7, 5, 9]`. At `8`, smaller child is `5`. Swap → `[3, 5, 7, 8, 9]`.

```
after: [3, 5, 7, 8, 9]

      3
     / \
    5   7
   / \
  8   9
```

**pop → 3.** Last `9` to root: `[9, 5, 7, 8]`. Swap with `5` → `[5, 9, 7, 8]`. Then `9` swaps with `8` → `[5, 8, 7, 9]`.

```
after: [5, 8, 7, 9]

      5
     / \
    8   7
   /
  9
```

**pop → 5.** Last `9` to root: `[9, 8, 7]`. Smaller child is `7`. Swap → `[7, 8, 9]`.

```
after: [7, 8, 9]

    7
   / \
  8   9
```

**pop → 7.** Last `9` to root: `[9, 8]`. Swap with `8` → `[8, 9]`.

```
after: [8, 9]

    8
   /
  9
```

**pop → 8.** Last `9` to root: `[9]`. No children → done.

```
after: [9]

    9
```

**pop → 9.** `top = 9`, `heap.pop()` empties the list. The `if heap:` block is skipped (no sift). Heap is `[]`.

Repeated pops give **sorted order** — that *is* heapsort (without the usual in-place version). Cost: `O(log n)` per push/pop, so building then emptying is `O(n log n)`.

#### Mental model

- **Complete tree in an array** → parent/child are arithmetic, not pointers.
- **push** = “put it in the next hole, walk **up** until it is not smaller than its parent.”
- **pop** = “hand back the root, drop the last leaf on the throne, walk **down** swapping with the **smaller** child until it is not larger than both children.”
- Stopping early (`break` / `small == i`) is the heap property being restored — you do not scan the whole array.

## 2. heapify — turn any array into a heap in O(n)

You already have an unordered list and want a valid heap **without** pushing items one by one (`O(n log n)`). **heapify** walks **internal nodes from the bottom up** and sifts each one down — leaves are already heaps of size 1, so you only fix parents. Most nodes sit near the bottom and barely move, so the total work is **`O(n)`**, not `O(n log n)`.

In [ ]:
def sift_down(a, i, n):
    while True:
        small, l, r = i, 2*i + 1, 2*i + 2
        if l < n and a[l] < a[small]: small = l
        if r < n and a[r] < a[small]: small = r
        if small == i:
            return
        a[i], a[small] = a[small], a[i]
        i = small

def heapify(a):
    """Bottom-up: sift-down every internal node. Total work is O(n), not O(n log n)."""
    n = len(a)
    for i in range(n // 2 - 1, -1, -1):
        sift_down(a, i, n)

arr = [9, 4, 7, 1, 3, 8, 2]
heapify(arr)
print("heapified:", arr)
assert arr[0] == min(arr)                        # root is the minimum

## 3. Python's `heapq` (min-heap) and the max-heap trick

Don't hand-roll a heap in interviews — `heapq` is Python's min-heap: `heappush` / `heappop` always surface the **smallest** item. To rank by something other than the value itself, push a **tuple** `(priority, item)` so the smallest priority comes out first. `heapq` has no max-heap; store **negatives** (`push -x`, pop then negate again) so the largest original number is treated as the smallest in the heap.

In [2]:
import heapq

h = []
for x in [5, 1, 3]:
    heapq.heappush(h, x)
print("heappop ->", heapq.heappop(h), "(the minimum)")

# order by an explicit key using tuples (priority, item) -- as Dijkstra does
pq = []
heapq.heappush(pq, (2, "task-B"))
heapq.heappush(pq, (1, "task-A"))
print("highest priority:", heapq.heappop(pq))    # (1, 'task-A')

# MAX-heap: negate on the way in and out
mx = []
for x in [5, 1, 3]:
    heapq.heappush(mx, -x)            # push negatives
largest = -heapq.heappop(mx)          # negate on the way out -> the maximum
print("max via negation:", largest)
assert largest == 5

heappop -> 1 (the minimum)
highest priority: (1, 'task-A')
max via negation: 5


## 4. Top-K with a size-K min-heap — O(n log k)

You only need the **K largest** values, not a full sort of all n numbers. Keep a min-heap of size at most K: push every number, and whenever the heap grows past K, **pop the smallest** so the heap is always “the K biggest seen so far” (its root is the weakest of those K). Each of the n pushes/pops costs `O(log K)`, so the whole pass is `O(n log K)` instead of `O(n log n)` sorting.

In [3]:
import heapq

def top_k(nums, k):
    """Keep a min-heap of size k; the k largest survive."""
    h = []
    for x in nums:
        heapq.heappush(h, x)
        if len(h) > k:
            heapq.heappop(h)                     # drop the smallest -> only top-k remain
    return sorted(h, reverse=True)

print("top 3 of [4,1,7,3,9,2,8]:", top_k([4, 1, 7, 3, 9, 2, 8], 3))
assert top_k([4, 1, 7, 3, 9, 2, 8], 3) == [9, 8, 7]

top 3 of [4,1,7,3,9,2,8]: [9, 8, 7]


## 5. Streaming median with two heaps — O(1) median

Numbers arrive one at a time and after each arrival you need the **median now** — you cannot re-sort the whole stream. Split the values into a **lower half** (max-heap, so its top is the largest small number) and an **upper half** (min-heap, so its top is the smallest large number), and keep the two heaps within one of each other in size. The median is then either the lower top (odd count) or the average of the two tops (even count) — **`O(1)`** to read, **`O(log n)`** to insert.

In [ ]:
import heapq

class MedianFinder:
    def __init__(self):
        self.lo = []          # max-heap (store negatives) -> lower half
        self.hi = []          # min-heap -> upper half

    def add(self, x):
        heapq.heappush(self.lo, -x)                        # tentatively to lower half
        heapq.heappush(self.hi, -heapq.heappop(self.lo))   # move its max to upper half
        if len(self.hi) > len(self.lo):                    # rebalance so lo >= hi in size
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def median(self):
        if len(self.lo) > len(self.hi):
            return -self.lo[0]                             # odd count -> middle of lower half
        return (-self.lo[0] + self.hi[0]) / 2              # even count -> average of middles

mf = MedianFinder()
for x in [1, 2, 3]:
    mf.add(x)
print("median of 1,2,3 :", mf.median())
mf.add(4)
print("median of 1,2,3,4:", mf.median())
assert mf.median() == 2.5

## ✅ Recap
- A heap is a **complete binary tree** in an array; **root = min** (or max).
- **push** = append + sift-up; **pop** = swap-last-to-root + sift-down; both `O(log n)`, peek `O(1)`.
- **heapify** builds a heap in **`O(n)`**.
- `heapq` is min-only (negate for max); store `(priority, item)` tuples.
- Powers **top-K**, **two-heap median**, Dijkstra/Prim, and schedulers.

Next: [`13_Tries`](../13_Tries/README.md).